# Lab 1 - A Simple Fingerprint Biometric System
 
**Name:** Hamza mohammed
 
**Student ID:** 2316234


In [ ]:
import os, json
import numpy as np
import matplotlib.pyplot as plt
import fp_tools as fp          # our helper file
 
DATA = "fingerprint_data"
files = sorted(os.listdir(DATA))
print("Number of images:", len(files))
print("First five:", files[:5])


In [ ]:
paths = [f"{DATA}/P01_{k}.png" for k in range(1, 5)] + [f"{DATA}/P02_{k}.png" for k in range(1, 5)]
fp.show_images(paths, cols=4)


In [ ]:
minutiae = fp.show_pipeline(f"{DATA}/P01_1.png")

In [ ]:
print("Table shape (rows, columns):", minutiae.shape)
print("First three minutiae [x, y, angle, type]:")
print(np.round(minutiae[:3], 2))
print("Ridge endings   :", int((minutiae[:, 3] == 0).sum()))
print("Bifurcations    :", int((minutiae[:, 3] == 1).sum()))


In [ ]:
database = {}
for person in range(1, 11):
    person_id = f"P{person:02d}"
    database[person_id] = fp.extract_template(f"{DATA}/{person_id}_1.png")
 
for person_id, template in database.items():
    print(person_id, "->", len(template), "minutiae")


In [ ]:
to_save = {pid: t.tolist() for pid, t in database.items()}
with open("templates.json", "w") as f:
    json.dump(to_save, f)
print("Saved", len(to_save), "templates to templates.json")


In [ ]:
probe_genuine = fp.extract_template(f"{DATA}/P01_2.png")
probe_impostor = fp.extract_template(f"{DATA}/P02_2.png")
 
score_g, pairs_g = fp.match(database["P01"], probe_genuine)
score_i, pairs_i = fp.match(database["P01"], probe_impostor)
print("Genuine  (P01 vs P01): paired =", len(pairs_g), " score = %.1f" % score_g)
print("Impostor (P01 vs P02): paired =", len(pairs_i), " score = %.1f" % score_i)


In [ ]:
fp.show_match(f"{DATA}/P01_1.png", f"{DATA}/P01_2.png", database["P01"], probe_genuine, pairs_g, score_g)

In [ ]:
THRESHOLD = 12
 
def verify(claimed_id, probe_path, threshold=THRESHOLD):
    probe = fp.extract_template(probe_path)
    score, pairs = fp.match(database[claimed_id], probe)
    decision = "ACCEPT" if score >= threshold else "REJECT"
    return decision, round(score, 1)
 
print("Real P03   :", verify("P03", f"{DATA}/P03_2.png"))
print("Real P03   :", verify("P03", f"{DATA}/P03_4.png"))
print("Impostor   :", verify("P03", f"{DATA}/P07_2.png"))
print("Impostor   :", verify("P03", f"{DATA}/S01_1.png"))


In [ ]:
def identify(probe_path, threshold=THRESHOLD):
    probe = fp.extract_template(probe_path)
    best_id, best_score = None, 0
    for person_id, template in database.items():
        score, _ = fp.match(template, probe)
        if score > best_score:
            best_id, best_score = person_id, score
    if best_score >= threshold:
        return best_id, round(best_score, 1)
    return "UNKNOWN", round(best_score, 1)
 
print(identify(f"{DATA}/P05_3.png"))
print(identify(f"{DATA}/P08_2.png"))
print(identify(f"{DATA}/S02_1.png"))


In [ ]:
genuine_scores, impostor_scores = [], []
for person in range(1, 11):
    pid = f"P{person:02d}"
    for k in (2, 3, 4):
        probe = fp.extract_template(f"{DATA}/{pid}_{k}.png")
        for other_id, template in database.items():
            score, _ = fp.match(template, probe)
            if other_id == pid:
                genuine_scores.append(score)
            else:
                impostor_scores.append(score)
 
genuine = np.array(genuine_scores)
impostor = np.array(impostor_scores)
print("Genuine comparisons :", len(genuine), "  average score = %.1f" % genuine.mean())
print("Impostor comparisons:", len(impostor), "  average score = %.1f" % impostor.mean())

In [ ]:
plt.figure(figsize=(8, 4.5))
plt.hist(impostor, bins=30, alpha=0.7, color="tomato", label="Impostor (different fingers)")
plt.hist(genuine, bins=30, alpha=0.7, color="seagreen", label="Genuine (same finger)")
plt.axvline(THRESHOLD, color="black", linestyle="--", label=f"Threshold = {THRESHOLD}")
plt.xlabel("Match score")
plt.ylabel("Number of comparisons")
plt.title("Genuine vs impostor scores")
plt.legend()
plt.savefig("score_histogram.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
print("Threshold   FNMR (%)   FMR (%)")
for t in [5, 8, 10, 12, 15, 20, 30, 40]:
    fnmr = 100 * np.mean(genuine < t)
    fmr = 100 * np.mean(impostor >= t)
    print(f"{t:9d}   {fnmr:8.1f}   {fmr:7.2f}")


In [ ]:
thresholds = np.arange(0, 61)
fmr_curve = [100 * np.mean(impostor >= t) for t in thresholds]
fnmr_curve = [100 * np.mean(genuine < t) for t in thresholds]
 
plt.figure(figsize=(8, 4.5))
plt.plot(thresholds, fmr_curve, color="tomato", label="FMR (impostor accepted)")
plt.plot(thresholds, fnmr_curve, color="seagreen", label="FNMR (genuine rejected)")
plt.xlabel("Threshold")
plt.ylabel("Error rate (%)")
plt.title("Error rates change with the threshold")
plt.legend()
plt.savefig("error_curve.png", dpi=150, bbox_inches="tight")
plt.show()


## Part I. Choose a threshold

**Phone unlock (convenience):** threshold = 8 because the False Match Rate (FMR) is higher, meaning more impostors may be accepted, but the False Non-Match Rate (FNMR) is lower, meaning fewer genuine users are rejected. This makes unlocking the phone more convenient.

**Bank vault (security):** threshold = 20 because the False Match Rate (FMR) is lower, meaning fewer impostors are accepted. Although the False Non-Match Rate (FNMR) is higher, meaning some genuine users may be rejected, security is more important than convenience when protecting valuable assets.

In [ ]:
img = fp.load_image(f"{DATA}/P01_3.png")
partial = fp.cover(img, 0.5)
plt.imshow(partial, cmap="gray"); plt.axis("off"); plt.show()
 
probe = fp.extract_template(partial)
score, pairs = fp.match(database["P01"], probe)
print("Half-covered genuine print: paired =", len(pairs), " score = %.1f" % score)
